In [1]:
import pandas as pd
import numpy as np

# ==========================
# CONFIG
# ==========================

GMV_FILE = "./raw_data/SM_HN_HCM_REV_filtered_2026_5.csv"
LEADS_FILE = "./raw_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv"

GMV_PHONE_COL = "Phone"
GMV_UID_COL = "UID"

LEADS_PHONE_COL = "Số điện thoại"
LEADS_UID_COL = "UID"


# ==========================
# CLEAN UID
# ==========================

def clean_uid(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

    s = s.replace("", np.nan)

    return s


# ==========================
# CLEAN PHONE
# ==========================

def clean_phone(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    # quá ngắn hoặc quá dài -> invalid
    s = s.where(s.str.len().between(9, 13), np.nan)

    return s


# ==========================
# LOAD
# ==========================

df_gmv = pd.read_csv(GMV_FILE)
df_leads = pd.read_csv(LEADS_FILE)


# ==========================
# CLEAN
# ==========================

df_gmv["UID_clean"] = clean_uid(df_gmv[GMV_UID_COL])
df_gmv["Phone_clean"] = clean_phone(df_gmv[GMV_PHONE_COL])

df_leads["UID_clean"] = clean_uid(df_leads[LEADS_UID_COL])
df_leads["Phone_clean"] = clean_phone(df_leads[LEADS_PHONE_COL])


# ==========================
# EXPORT
# ==========================

df_gmv.to_csv(
    "./cleaned_data/gmv_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_leads.to_csv(
    "./cleaned_data/leads_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Done!


In [ ]:
df_gmv = pd.read_csv("./cleaned_data/gmv_clean.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding='utf-8-sig')

In [ ]:
df_gmv.head()

In [4]:
print(df_gmv["TEAM"].unique())

<StringArray>
['HCM', 'In-house', 'In-house 2', 'Linh Dam Store']
Length: 4, dtype: str


In [ ]:
unique_prefix = (
    df_leads["TVTS"]
    .dropna()
    .str.split("-", n=1)
    .str[0]
    .str.strip()
    .unique()
)

print(unique_prefix)

['HN' 'HN2' 'HCM' 'OFF' 'HuyenKTT' 'LyBP' 'AnhVP' 'kho' 'Phuong' 'tr'
 'IND' 'TamNTT' 'TrangDT' 'LinhNTT' 'ThinhND']


In [15]:
# Drop NaN values to avoid errors during string operations
tvts_series = df_leads["TVTS"].dropna()

# Strip spaces and filter the Series for values that start with 'OFF'
off_values = tvts_series[tvts_series.str.strip().str.startswith("OFF")]

# If you want to see all occurrences
print(len(off_values))

# If you only want the unique 'OFF - something' string values
unique_off_values = off_values.unique()
print(unique_off_values)

50
<StringArray>
['OFF - Julia']
Length: 1, dtype: str


In [32]:
import pandas as pd
import numpy as np

df_gmv = pd.read_csv("./test/gmv_clean.csv", encoding="utf-8-sig")

# 1. Define the exact mapping for the 4 TEAM cases
team_mapping = {
    "In-house": "HN",
    "In-house 2": "HN2",
    "HCM": "HCM",
    "Linh Dam Store": "OFF"
}

# 2. Function to automatically format Vietnamese names
def format_vietnamese_name(name):
    # Handle empty or NaN values
    if pd.isna(name):
        return ""
    
    # Strip leading/trailing spaces
    name = str(name).strip()
    parts = name.split()
    
    # If the name has spaces (e.g., "Cao Thi Lua"), convert to "LuaCT"
    if len(parts) > 1:
        # Take the last word as the main name
        first_name = parts[-1]
        # Get the first letter of all preceding words and uppercase them
        initials = "".join([p[0].upper() for p in parts[:-1]])
        return f"{first_name}{initials}"
    
    # If it's already a single string without spaces (e.g., "LinhNT"), return as is
    return name

# 3. Create the new column for abbreviated Sales names ONLY
df_gmv["Sales_Abbr"] = df_gmv["Sales"].apply(format_vietnamese_name)

# 4. Apply the team mapping to get the corresponding prefixes
prefixes = df_gmv["TEAM"].map(team_mapping)

# 5. Define conditions and choices for the 'Sales_infor' column
conditions = [
    # Condition 1: If TEAM is Linh Dam Store, hardcode the value to 'OFF - Julia'
    df_gmv["TEAM"] == "Linh Dam Store",
    
    # Condition 2: For other mapped teams, combine prefix and the abbreviated name
    prefixes.notna()
]

choices = [
    # Output for Condition 1
    "OFF - Julia",
    
    # Output for Condition 2
    prefixes + " - " + df_gmv["Sales_Abbr"]
]

# 6. Apply np.select to create the 'Sales_infor' column based on the conditions
# Teams not in the mapping will get NaN
df_gmv["Sales_infor"] = np.select(conditions, choices, default=np.nan)

# Print a sample to verify both new columns
print(df_gmv[["TEAM", "Sales", "Sales_Abbr", "Sales_infor"]])

# Export to CSV
df_gmv.to_csv('./test/gmv_clean_sales_infor.csv', encoding='utf-8-sig', index=False)

           TEAM               Sales Sales_Abbr     Sales_infor
0           HCM    Nguyen Minh Phat     PhatNM    HCM - PhatNM
1           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
2           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
3           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
4           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
..          ...                 ...        ...             ...
458    In-house  Luu Thi Hoang Ngan    NganLTH    HN - NganLTH
459    In-house  Luu Thi Hoang Ngan    NganLTH    HN - NganLTH
460    In-house   Nguyen Kieu Trang    TrangNK    HN - TrangNK
461  In-house 2   Vu Ho Thanh Huong   HuongVHT  HN2 - HuongVHT
462  In-house 2   Vu Ho Thanh Huong   HuongVHT  HN2 - HuongVHT

[463 rows x 4 columns]


In [17]:
# Group the dataframe by 'TEAM' and extract the first 3 rows of each group
# We only select the relevant columns for a cleaner output
sample_check = df_gmv[["TEAM", "Sales", "Sales_Abbr", "Sales_infor"]].groupby("TEAM").head(3)

# Display the result to verify the applied logic
print(sample_check)

               TEAM               Sales Sales_Abbr    Sales_infor
0               HCM    Nguyen Minh Phat     PhatNM   HCM - PhatNM
1               HCM  Lai Ngoc Thuy Linh    LinhLNT  HCM - LinhLNT
2               HCM  Lai Ngoc Thuy Linh    LinhLNT  HCM - LinhLNT
19         In-house         Cao Thi Lua      LuaCT     HN - LuaCT
20         In-house        Le Thi Tuyet    TuyetLT   HN - TuyetLT
21         In-house   Pham Thi Tam Thuy    ThuyPTT   HN - ThuyPTT
34       In-house 2         Le Thi Tram     TramLT   HN2 - TramLT
35       In-house 2           Vu Cam Ly       LyVC     HN2 - LyVC
48       In-house 2           Vu Cam Ly       LyVC     HN2 - LyVC
56   Linh Dam Store   Ngo Thi Thuy Linh    LinhNTT    OFF - Julia
123  Linh Dam Store      Nguyen Thi Lan      LanNT    OFF - Julia
127  Linh Dam Store      Nguyen Thi Lan      LanNT    OFF - Julia


In [9]:
import pandas as pd

df_gmv = pd.read_csv('./cleaned_data/gmv_clean_sales_infor.csv', encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding='utf-8-sig')

In [10]:
print(df_gmv[["UID_clean", "Phone_clean"]].head(10))
print(df_leads[["UID_clean", "Phone_clean"]].head(10))

    UID_clean  Phone_clean
0  3311001921  84908224672
1  3179205818  84909517679
2  3311304274  84964678701
3  3310902627  84389970625
4  3309271098  84938593961
5  3311605431  84939899045
6  3310167280  84978827804
7  3311379165  84948063983
8  3311850607  84982135774
9  3311951330  84938222653
    UID_clean   Phone_clean
0         NaN           NaN
1  3310947243  8.491925e+10
2  3310952712  8.496547e+10
3  3293618952  8.498595e+10
4  3310947249  8.497286e+10
5  3310952706  8.490728e+10
6  3310947256  8.438888e+10
7  3310947803  8.496846e+10
8  3310947816  8.491491e+10
9  3310947809  8.438767e+10


In [4]:
import pandas as pd

# Đọc dữ liệu
df_gmv = pd.read_csv("./cleaned_data/gmv_clean.csv", encoding="utf-8-sig")
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding="utf-8-sig")

# ==========================
# Chuẩn hóa dữ liệu
# ==========================
for df in [df_gmv, df_leads]:
    for col in ["UID_clean", "Phone_clean"]:
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.replace(".0", "", regex=False)   # bỏ .0
            .str.strip()                          # bỏ khoảng trắng
        )

# (Tùy chọn) Loại bỏ dòng mà cả UID và Phone đều rỗng
df_gmv = df_gmv[
    (df_gmv["UID_clean"] != "") |
    (df_gmv["Phone_clean"] != "")
]

df_leads = df_leads[
    (df_leads["UID_clean"] != "") |
    (df_leads["Phone_clean"] != "")
]

# ==========================
# Tạo set (UID, Phone)
# ==========================
gmv_pairs = set(zip(df_gmv["UID_clean"], df_gmv["Phone_clean"]))
leads_pairs = set(zip(df_leads["UID_clean"], df_leads["Phone_clean"]))

# Giao nhau
common_pairs = gmv_pairs & leads_pairs

# ==========================
# Kết quả
# ==========================
print(f"Số cặp (UID, Phone) trong GMV   : {len(gmv_pairs)}")
print(f"Số cặp (UID, Phone) trong Leads : {len(leads_pairs)}")
print(f"Số cặp giao nhau                : {len(common_pairs)}")

print("\n20 cặp giao nhau đầu tiên:")
for pair in list(common_pairs)[:20]:
    print(pair)

# ==========================
# Debug thêm
# ==========================
uid_common = set(df_gmv["UID_clean"]) & set(df_leads["UID_clean"])
phone_common = set(df_gmv["Phone_clean"]) & set(df_leads["Phone_clean"])

print("\n==============================")
print(f"UID common   : {len(uid_common)}")
print(f"Phone common : {len(phone_common)}")

Số cặp (UID, Phone) trong GMV   : 457
Số cặp (UID, Phone) trong Leads : 15828
Số cặp giao nhau                : 259

20 cặp giao nhau đầu tiên:
('3312030706', '84365684099')
('3311063583', '84369140500')
('3311542483', '84983288462')
('3230039800', '84936141588')
('3311363080', '84982939616')
('3311026002', '84989864468')
('3309549687', '84865800589')
('3311109861', '84981763246')
('3311997856', '818075791989')
('3306006558', '819051507790')
('3311090150', '84908594887')
('3310653692', '84978876305')
('3311055427', '84919729288')
('3311195799', '84975464090')
('3310799395', '819056896069')
('3311711014', '84935694353')
('3281837023', '4915252600892')
('3312001679', '819017286246')
('3311287650', '84386984330')
('3312006651', '84902377230')

UID common   : 259
Phone common : 278


In [16]:
import pandas as pd

# ==========================
# Read data
# ==========================
df_gmv = pd.read_csv("./cleaned_data/gmv_clean_sales_infor.csv", encoding="utf-8-sig")
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding="utf-8-sig")

# ==========================
# Normalize
# ==========================

# GMV
for col in ["UID_clean", "Phone_clean", "Sales_infor"]:
    df_gmv[col] = (
        df_gmv[col]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Leads
for col in ["UID_clean", "Phone_clean", "TVTS"]:
    df_leads[col] = (
        df_leads[col]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Loại bỏ dòng không có cả UID và Phone
df_gmv = df_gmv[
    (df_gmv["UID_clean"] != "") |
    (df_gmv["Phone_clean"] != "")
].copy()

df_leads = df_leads[
    (df_leads["UID_clean"] != "") |
    (df_leads["Phone_clean"] != "")
].copy()

# ==========================
# Build result
# ==========================
matched_rows = []
unmatched_rows = []

matched_count = 0

# Các cột của leads sẽ được nối thêm vào GMV
lead_columns = list(df_leads.columns)

for idx, gmv_row in df_gmv.iterrows():

    uid = gmv_row["UID_clean"]
    phone = gmv_row["Phone_clean"]
    sales = gmv_row["Sales_infor"]

    # --------------------------
    # Rule 1: UID + Phone
    # --------------------------
    candidates = df_leads[
        (df_leads["UID_clean"] == uid)
        &
        (df_leads["Phone_clean"] == phone)
    ]

    reason = ""

    if len(candidates) == 0:
        # Không match
        new_row = gmv_row.to_dict()

        for col in lead_columns:
            new_row[f"Lead_{col}"] = pd.NA

        new_row["Match_Status"] = "Unmatched"
        new_row["Match_Reason"] = "No UID+Phone"

        unmatched_rows.append(new_row)

    else:

        if len(candidates) == 1:

            chosen = candidates.iloc[0]
            reason = "Unique UID+Phone"

        else:

            # Ưu tiên TVTS == Sales_infor
            same_sales = candidates[
                candidates["TVTS"] == sales
            ]

            if len(same_sales) > 0:

                chosen = same_sales.iloc[0]
                reason = "Matched by Sales_infor"

            else:

                chosen = candidates.iloc[0]
                reason = "First candidate"

        new_row = gmv_row.to_dict()

        for col in lead_columns:
            new_row[f"Lead_{col}"] = chosen[col]

        new_row["Match_Status"] = "Matched"
        new_row["Match_Reason"] = reason

        matched_rows.append(new_row)
        matched_count += 1

# ==========================
# Export
# ==========================

matched_df = pd.DataFrame(matched_rows)
unmatched_df = pd.DataFrame(unmatched_rows)

final_df = pd.concat(
    [matched_df, unmatched_df],
    ignore_index=True
)

final_df.to_csv(
    "./output/final_join.csv",
    index=False,
    encoding="utf-8-sig"
)

matched_df.to_csv(
    "./output/matched.csv",
    index=False,
    encoding="utf-8-sig"
)

unmatched_df.to_csv(
    "./output/unmatched.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================
# Statistics
# ==========================
print("=" * 50)
print(f"GMV rows      : {len(df_gmv):,}")
print(f"Matched       : {matched_count:,}")
print(f"Unmatched     : {len(unmatched_df):,}")
print(f"Match rate    : {matched_count/len(df_gmv):.2%}")
print("=" * 50)

print("\nReason:")
print(final_df["Match_Reason"].value_counts())

GMV rows      : 463
Matched       : 261
Unmatched     : 202
Match rate    : 56.37%

Reason:
Match_Reason
Unique UID+Phone          253
No UID+Phone              202
Matched by Sales_infor      5
First candidate             3
Name: count, dtype: int64


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14360\1439376757.py:126: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(


In [17]:
# Chuẩn hóa trước
for df in [df_gmv, df_leads]:
    df["Phone_clean"] = (
        df["Phone_clean"]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Tạo set
gmv_phones = set(df_gmv["Phone_clean"])
leads_phones = set(df_leads["Phone_clean"])

# Giao nhau
common_phones = gmv_phones & leads_phones

print(f"GMV phones    : {len(gmv_phones)}")
print(f"Leads phones  : {len(leads_phones)}")
print(f"Common phones : {len(common_phones)}")

GMV phones    : 457
Leads phones  : 15672
Common phones : 278


In [18]:


# Chuẩn hóa
for df in [df_gmv, df_leads]:
    df["Phone_clean"] = (
        df["Phone_clean"]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Tập phone của leads
lead_phone_set = set(df_leads["Phone_clean"])

# Đánh dấu match
df_gmv["Phone_Matched"] = df_gmv["Phone_clean"].isin(lead_phone_set)

# Thống kê
matched_count = df_gmv["Phone_Matched"].sum()
unmatched_count = len(df_gmv) - matched_count

print("=" * 40)
print(f"GMV rows   : {len(df_gmv)}")
print(f"Matched    : {matched_count}")
print(f"Unmatched  : {unmatched_count}")
print(f"Match rate : {matched_count / len(df_gmv):.2%}")
print("=" * 40)

GMV rows   : 463
Matched    : 280
Unmatched  : 183
Match rate : 60.48%


In [19]:
# Chuẩn hóa
for df in [df_gmv, df_leads]:
    df["UID_clean"] = (
        df["UID_clean"]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Tập UID của Leads
lead_uid_set = set(df_leads["UID_clean"])

# Đánh dấu match
df_gmv["UID_Matched"] = df_gmv["UID_clean"].isin(lead_uid_set)

# Thống kê
matched_count = df_gmv["UID_Matched"].sum()
unmatched_count = len(df_gmv) - matched_count

print("=" * 40)
print(f"GMV rows   : {len(df_gmv)}")
print(f"Matched    : {matched_count}")
print(f"Unmatched  : {unmatched_count}")
print(f"Match rate : {matched_count / len(df_gmv):.2%}")
print("=" * 40)

# (Tuỳ chọn) Lấy các dòng match / không match
matched_df = df_gmv[df_gmv["UID_Matched"]]
unmatched_df = df_gmv[~df_gmv["UID_Matched"]]

GMV rows   : 463
Matched    : 261
Unmatched  : 202
Match rate : 56.37%


In [20]:
# Chuẩn hóa
for df in [df_gmv, df_leads]:
    df["Phone_clean"] = (
        df["Phone_clean"]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Tập phone của Leads
lead_phone_set = set(df_leads["Phone_clean"])

# Đánh dấu match
df_gmv["Phone_Matched"] = df_gmv["Phone_clean"].isin(lead_phone_set)

# Lấy những dòng không match
unmatched_df = df_gmv[~df_gmv["Phone_Matched"]].copy()

# Xuất file
unmatched_df.to_csv(
    "./output/unmatched_by_phone.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"GMV rows      : {len(df_gmv)}")
print(f"Unmatched rows: {len(unmatched_df)}")
print("Saved to: ./output/unmatched_by_phone.csv")

GMV rows      : 463
Unmatched rows: 183
Saved to: ./output/unmatched_by_phone.csv


In [21]:
import pandas as pd

# Define the file path (replace 'your_file.csv' with your actual file name)
file_path = './output/unmatched_by_phone.csv'

# Read the data into a pandas DataFrame
# Note: If your file is an Excel file, use pd.read_excel('your_file.xlsx')
df = pd.read_csv(file_path)

# List of specific columns you want to extract and print
# I removed the leading comma from your list to make it a valid Python list
columns_to_print = [
    'UID_clean', 
    'Phone_clean', 
    'Sales_infor', 
    'Sales_Abbr', 
    'Phone_Matched', 
    'UID_Matched'
]

# Print the selected columns
print(df[columns_to_print])

      UID_clean   Phone_clean     Sales_infor Sales_Abbr  Phone_Matched  \
0    3311001921   84908224672    HCM - PhatNM     PhatNM          False   
1    3179205818   84909517679   HCM - LinhLNT    LinhLNT          False   
2    3310902627   84389970625   HCM - LinhLNT    LinhLNT          False   
3    3311951330   84938222653   HCM - LinhLNT    LinhLNT          False   
4    3311919278   84844534222    HCM - PhatNM     PhatNM          False   
..          ...           ...             ...        ...            ...   
178  3180717063   84368776525   HN - TrangLTT   TrangLTT          False   
179  3310177620  817035351368    HN - NganLTH    NganLTH          False   
180  3311996848   84969663003    HN - TrangNK    TrangNK          False   
181  3312285639   84943111987  HN2 - HuongVHT   HuongVHT          False   
182  3312285639   84943111987  HN2 - HuongVHT   HuongVHT          False   

     UID_Matched  
0          False  
1          False  
2          False  
3          False  
4   

In [23]:
import re
# Đảm bảo Note không bị NaN
df_leads["Note"] = df_leads["Note"].fillna("").astype(str)

# Regex tìm chuỗi giống số điện thoại
pattern = r'[\d\-\s().]{9,20}'

results = []

for idx, note in df_leads["Note"].items():

    matches = re.findall(pattern, note)

    for m in matches:

        # Chỉ giữ lại số
        phone = re.sub(r"\D", "", m)

        # Giữ các số có độ dài hợp lý
        if 9 <= len(phone) <= 12:

            results.append({
                "Row": idx,
                "Extracted_Phone": phone,
                "Original_Text": m.strip(),
                "Note": note
            })

phones_df = pd.DataFrame(results)

print(f"Extracted phones: {len(phones_df)}")
print(f"Unique phones: {phones_df['Extracted_Phone'].nunique()}")

phones_df.to_csv(
    "./output/extracted_phones_from_note.csv",
    index=False,
    encoding="utf-8-sig"
)

Extracted phones: 510
Unique phones: 459


In [25]:
df_phone_extract = pd.read_csv('./output/extracted_phones_from_note.csv', encoding='utf-8')
print(df_phone_extract['Extracted_Phone'])

0         965468525
1         985951781
2         933825253
3         962340965
4         978829218
           ...     
505    821055905677
506    821099617677
507       969954417
508     84367740544
509     84467298738
Name: Extracted_Phone, Length: 510, dtype: int64


In [26]:
import pandas as pd

# Đọc file
df_phone_extract = pd.read_csv(
    "./output/extracted_phones_from_note.csv",
    encoding="utf-8"
)

# Chuẩn hóa
df_gmv["Phone_clean"] = (
    df_gmv["Phone_clean"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

df_phone_extract["Extracted_Phone"] = (
    df_phone_extract["Extracted_Phone"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# Tạo set
gmv_phone_set = set(df_gmv["Phone_clean"])
extract_phone_set = set(df_phone_extract["Extracted_Phone"])

# Giao nhau
common_phone = gmv_phone_set & extract_phone_set

print(f"GMV phones              : {len(gmv_phone_set)}")
print(f"Extracted phones        : {len(extract_phone_set)}")
print(f"Common phones           : {len(common_phone)}")

print("\n20 phone đầu tiên:")
for phone in list(common_phone)[:20]:
    print(phone)

GMV phones              : 457
Extracted phones        : 459
Common phones           : 9

20 phone đầu tiên:
819091229192
818090801510
84367707624
84338692677
491727155802
817090293668
821047062407
817084801084
84916609147


In [27]:
# Giao giữa GMV và Leads
common_gmv_leads = gmv_phone_set & lead_phone_set

# Giao giữa GMV và Extracted Phone
common_gmv_extract = gmv_phone_set & extract_phone_set

# Những phone mới được tìm thấy từ Note
new_phones = common_gmv_extract - common_gmv_leads

print(f"GMV ∩ Leads           : {len(common_gmv_leads)}")
print(f"GMV ∩ Extract         : {len(common_gmv_extract)}")
print(f"New phones recovered  : {len(new_phones)}")

print("\nNew phones:")
for p in sorted(new_phones):
    print(p)

GMV ∩ Leads           : 278
GMV ∩ Extract         : 9
New phones recovered  : 3

New phones:
819091229192
84338692677
84916609147


In [28]:
# Chuẩn hóa
for df in [df_gmv, df_leads]:
    df["UID_clean"] = (
        df["UID_clean"]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

df_phone_extract["Extracted_Phone"] = (
    df_phone_extract["Extracted_Phone"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# Tạo set
gmv_uid_set = set(df_gmv["UID_clean"])
lead_uid_set = set(df_leads["UID_clean"])
extract_phone_set = set(df_phone_extract["Extracted_Phone"])

# Giao
gmv_uid_match = gmv_uid_set & extract_phone_set
lead_uid_match = lead_uid_set & extract_phone_set

print(f"GMV UID ∩ Extracted Phone   : {len(gmv_uid_match)}")
print(f"Lead UID ∩ Extracted Phone  : {len(lead_uid_match)}")

print("\nGMV UID match:")
for x in list(gmv_uid_match)[:20]:
    print(x)

print("\nLead UID match:")
for x in list(lead_uid_match)[:20]:
    print(x)

GMV UID ∩ Extracted Phone   : 0
Lead UID ∩ Extracted Phone  : 14

GMV UID match:

Lead UID match:
3304865814
3307556253
3307456092
3312218325
3304621312
3307137324
3305021132
3305765039
3299168771
3307371607
3307345276
3306104073
3310067499
3308813596


In [31]:
import re

df_leads = pd.read_csv("./test/leads_clean.csv")
# Đảm bảo Note không bị NaN
df_leads["Note"] = df_leads["Note"].fillna("").astype(str)

# Regex tìm chuỗi giống số điện thoại
pattern = r'[\d\-\s().]{9,20}'

def extract_phones(note):
    matches = re.findall(pattern, note)

    phones = []

    for m in matches:
        # Chỉ giữ lại chữ số
        phone = re.sub(r"\D", "", m)

        # Chỉ giữ các số có độ dài hợp lý
        if 9 <= len(phone) <= 12:
            phones.append(phone)

    # Loại bỏ trùng nhưng vẫn giữ thứ tự
    phones = list(dict.fromkeys(phones))

    return ";".join(phones)

# Thêm cột mới
df_leads["Phone_extracted_from_note"] = df_leads["Note"].apply(extract_phones)

# Thống kê
num_rows = (df_leads["Phone_extracted_from_note"] != "").sum()

print(f"Rows containing extracted phone: {num_rows}")

# Lưu lại nếu muốn
df_leads.to_csv(
    "./test/leads_clean_with_note_phone.csv",
    index=False,
    encoding="utf-8-sig"
)

Rows containing extracted phone: 437


KeyError: 'match_type'